# DDPM with FashionMNIST - Compact Implementation

A streamlined DDPM training notebook using the ddpm module architecture applied to FashionMNIST dataset.

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

#import sys
#sys.path.insert(0, '.')  # Current directory

import os
start_dir = os.getcwd()
os.chdir('/Users/vivienohlms/Documents/UvA/Period 5/Machine Learning/diffusion-models-project')

from ddpm import NoiseScheduler, UNet, generate_image
from ddpm.dataset import NoisyMNIST
from ddpm.utils import channel_list, model_name, path_name
from ddpm import train, NoisyDataset
os.chdir(start_dir)

from torch.optim import Adam

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


## Step 1: Load and Preprocess FashionMNIST Data

In [ ]:
# Load FashionMNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.FashionMNIST('../data', train=True, transform=transform, download=True)
test_dataset = datasets.FashionMNIST('../data', train=False, transform=transform, download=True)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

print(f"Training set: {len(train_dataset)} images")
print(f"Test set: {len(test_dataset)} images")
print(f"Image shape: {train_dataset[0][0].shape}")

Training set: 60000 images
Test set: 10000 images
Image shape: torch.Size([1, 28, 28])


## Step 2: Setup Noise Schedule

In [15]:
# Initialize noise scheduler with 1000 timesteps
scheduler = NoiseScheduler(T=1000, beta_start=1e-4, beta_end=0.02)

# Create noisy datasets
train_set_noisy = NoisyMNIST(train_dataset, scheduler)
test_set_noisy = NoisyMNIST(test_dataset, scheduler)

print(f"Noise scheduler initialized with T={scheduler.T} timesteps")

Noise scheduler initialized with T=1000 timesteps


## Step 3: Build U-Net Architecture

Configure the model with channel depth and convolutions per level

In [16]:
# Model configuration
channel0 = 64  # Base channel depth
convs_per_level = 2  # Convolutions per resolution level

# Build channel list (automatically creates encoder/decoder structure)
channels = channel_list(channel0)

# Create U-Net model
unet = UNet(channels=channels, convs_per_level=convs_per_level).to(device)

print(f"Model: {model_name(channel0, convs_per_level)}")
print(f"Parameters: {sum(p.numel() for p in unet.parameters()):,}")

Model: C0_64_convs_2
Parameters: 1,881,985


## Step 4: Training Setup

Configure optimizer and training parameters

In [17]:
# Training hyperparameters
learning_rate = 1e-3
weight_decay = 1e-6
epochs = 20
early_stopping_patience = 5

optimizer = Adam(unet.parameters(), lr=learning_rate, weight_decay=weight_decay)
loss_fn = nn.MSELoss()

print(f"Learning rate: {learning_rate}")
print(f"Weight decay: {weight_decay}")
print(f"Epochs: {epochs}")

Learning rate: 0.001
Weight decay: 1e-06
Epochs: 20


## Step 5: Train the Diffusion Model

Train locally (small epochs for demonstration)

In [ ]:
# Create noisy data loaders (num_workers=0 to avoid multiprocessing issues)
train_noisy_loader = torch.utils.data.DataLoader(
    train_set_noisy, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True
)
test_noisy_loader = torch.utils.data.DataLoader(
    test_set_noisy, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True
)

# Train locally
train_losses, test_losses = train(
    unet, train_noisy_loader, test_noisy_loader,
    epochs=epochs,
    lr=learning_rate,
    weight_decay=weight_decay,
    early_stopping_patience=early_stopping_patience,
    save_path=path_name(channel0, convs_per_level, add_desc="fashion_mnist"),
)

print("Training complete!")

Epochs:   0%|          | 0/20 [00:00<?, ?it/s]

train:   0%|          | 0/1875 [00:00<?, ?it/s]

/opt/anaconda3/envs/ddpm/lib/python3.11/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


test:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 0 | train loss: 0.0600 | test loss: 0.0398


train:   0%|          | 0/1875 [00:00<?, ?it/s]

test:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 1 | train loss: 0.0358 | test loss: 0.0324


train:   0%|          | 0/1875 [00:00<?, ?it/s]

test:   0%|          | 0/313 [00:00<?, ?it/s]

## Step 6: Visualize Training Progress

In [ ]:
def plot_loss(train_losses, test_losses):
    """Plot training and test loss curves."""
    plt.figure(figsize=(10, 4))
    plt.plot(train_losses, label='Train Loss', linewidth=2)
    plt.plot(test_losses, label='Test Loss', linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('DDPM Training on FashionMNIST')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_loss(train_losses, test_losses)

## Step 7: Generate Images from Trained Model

In [ ]:
from ddpm.viz import plot_generated

# Set model to eval mode
unet.eval()

# Generate images
n_images = 16
with torch.no_grad():
    generated = generate_image(unet, scheduler, stochasticity=1.0, n_images=n_images)

print(f"Generated {n_images} images")

# Visualize
plot_generated(generated, ncol=4)

## Step 8: Compare Generated vs Ground Truth

In [ ]:
# Get real test samples
real_images = []
for batch in test_loader:
    real_images.append(batch[0])
    if len(real_images) * batch_size >= n_images:
        break
real_images = torch.cat(real_images, dim=0)[:n_images]

print("Real FashionMNIST samples:")
plot_generated(real_images, ncol=4)